# NHANES Cholesterol


In [ ]:
from pathlib import Path
import pickle

import pandas as pd

try:
    from experiment import (
        NHANES_ARTHRITIS_SOURCES,
        NHANES_ARTHRITIS_TARGET,
        NHANES_TEMPORAL_SOURCES,
        NHANES_TEMPORAL_TARGET,
        experiment_config,
        prepare_nhanes,
        procedure_sha256,
        run_trials,
        save_results,
        summarize_results,
    )
    from plotting import plot_comparison, plot_target_qq
except ImportError:
    from dataset_ranking.experiment import (
        NHANES_ARTHRITIS_SOURCES,
        NHANES_ARTHRITIS_TARGET,
        NHANES_TEMPORAL_SOURCES,
        NHANES_TEMPORAL_TARGET,
        experiment_config,
        prepare_nhanes,
        procedure_sha256,
        run_trials,
        save_results,
        summarize_results,
    )
    from dataset_ranking.plotting import plot_comparison, plot_target_qq

## Prepare data

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
RESULT_DIR = ROOT / "results"
FIGURE_DIR = ROOT / "figures"

def validate_payload(payload, prepared_data):
    expected = experiment_config(
        prepared_data, prepared_data["seed_start"], prepared_data["trials"],
        prepared_data["n_test"], prepared_data["n_t"],
        prepared_data["n_k"], prepared_data.get("n_target_mean"),
    )
    assert payload["code_sha256"] == expected["code_sha256"]
    assert payload["score_x_definition"] == expected["score_x"]
    assert payload["duc_data"] == "standardized_then_whitened"
    assert all(
        payload["config"][key] == value
        for key, value in expected.items() if key != "environment"
    )
    assert all(
        result["run_config_sha256"] == procedure_sha256(payload["config"])
        for result in payload["results"]
    )

data = prepare_nhanes(DATA_DIR)
print("Target:", data["target_name"])
pd.DataFrame({"label": data["labels"], "dataset": data["source_names"]})


## Run experiment

In [ ]:
n_t = 30
n_k = 1000
n_test = 1000
trials = 1000
global_seed = 12000
jobs = 1

data.update({
    "trials": trials,
    "seed_start": global_seed,
    "n_test": n_test,
    "n_t": n_t,
    "n_k": n_k,
})

result_path = RESULT_DIR / "nhanes_cholesterol_results.pkl"
if result_path.exists():
    with open(result_path, "rb") as file:
        payload = pickle.load(file)
    assert payload["source_names"] == data["source_names"]
    assert payload["target_name"] == data["target_name"]
    assert payload["seeds"] == list(range(global_seed, global_seed + trials))
    assert payload["data_sha256"] == data["data_sha256"]
    validate_payload(payload, data)
    results = payload["results"]
else:
    results = run_trials(
        data,
        trials=trials,
        seed_start=global_seed,
        jobs=jobs,
        n_test=n_test,
        n_t=n_t,
        n_k=n_k,
        checkpoint_path=RESULT_DIR / "nhanes_cholesterol_checkpoint.pkl",
        checkpoint_every=4,
    )
    save_results(data, results, RESULT_DIR)
summary = summarize_results(results, data["labels"])
summary


## QQ plot

In [ ]:
plot_target_qq(
    data["qq_data"],
    data["target_id"],
    data["qq_source_ids"],
    data["site_column"],
    data["response"],
    labels=data["qq_labels"],
    target_label="Target",
    title="Normal QQ Plots: NHANES Cholesterol",
    path=FIGURE_DIR / "qqplot_target_vs_sources_nhanes.png",
)


## Result plots

In [ ]:
highlights = {
    "duc": {"D3"},
    "neg_kl": {"D5"},
    "neg_score_x": {"D2"},
}
red_highlights = {name: {"D4"} for name in highlights}
rank_label_offsets = {
    "neg_kl": {"D4": (-8, 10), "D6": (8, 10)},
    "neg_score_x": {"D4": (-8, 10)},
}
mse_label_offsets = {
    "neg_score_x": {
        "D2": (-14, 8),
        "D4": (-14, 12),
    },
}

plot_comparison(
    summary,
    "Comparison of methods for data selection: NHANES Cholesterol",
    y_column="avg_rank",
    y_label="Average Rank of Test MSE (1 = Best)",
    highlight_map=highlights,
    red_highlight_map=red_highlights,
    label_offsets=rank_label_offsets,
    corr_y=0.87,
    path=FIGURE_DIR / "fig_nhanes.png",
)

plot_comparison(
    summary,
    "Comparison of methods for data selection: NHANES Cholesterol",
    y_column="weighted_mse",
    y_label="Average Test MSE",
    highlight_map=highlights,
    label_offsets=mse_label_offsets,
    path=FIGURE_DIR / "fig_nhanes_mse.png",
)


In [ ]:
# Pure temporal shift: 2017 target and six prior-year source cuts
temporal_data = prepare_nhanes(
    DATA_DIR,
    source_names=NHANES_TEMPORAL_SOURCES,
    target_name=NHANES_TEMPORAL_TARGET,
)
temporal_data.update({
    "name": "nhanes_temporal",
    "trials": 1000,
    "seed_start": 14000,
    "n_test": 1000,
    "n_t": 30,
    "n_k": 1000,
})
temporal_path = RESULT_DIR / "nhanes_temporal_results.pkl"

if temporal_path.exists():
    with open(temporal_path, "rb") as file:
        temporal_payload = pickle.load(file)
    assert temporal_payload["source_names"] == NHANES_TEMPORAL_SOURCES
    assert temporal_payload["target_name"] == NHANES_TEMPORAL_TARGET
    assert temporal_payload["seeds"] == list(range(14000, 15000))
    assert temporal_payload["data_sha256"] == temporal_data["data_sha256"]
    validate_payload(temporal_payload, temporal_data)
    temporal_results = temporal_payload["results"]
else:
    temporal_results = run_trials(
        temporal_data,
        trials=1000,
        seed_start=14000,
        jobs=1,
        n_test=1000,
        n_t=30,
        n_k=1000,
        checkpoint_path=RESULT_DIR / "nhanes_temporal_checkpoint.pkl",
        checkpoint_every=4,
    )
    save_results(temporal_data, temporal_results, RESULT_DIR)

temporal_summary = summarize_results(temporal_results, temporal_data["labels"])
temporal_plot_summary = pd.read_csv(RESULT_DIR / "nhanes_temporal_summary.csv")
pd.testing.assert_frame_equal(
    temporal_summary,
    temporal_plot_summary,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)
temporal_figure = plot_comparison(
    temporal_plot_summary,
    "Comparison of methods for data selection: NHANES Cholesterol (temporal shift)",
    y_column="avg_rank",
    y_label="Average Rank of Test MSE (1 = Best)",
    corr_y=0.84,
    label_offsets={
        "duc": {"D2": (8, -8), "D4": (8, 8)},
        "neg_score_x": {"D2": (8, -8), "D4": (8, 8)},
    },
    path=FIGURE_DIR / "nhanes_temporal.png",
)

In [ ]:
# QQ plots for the pure temporal shift
temporal_qq_figure = plot_target_qq(
    temporal_data["qq_data"],
    temporal_data["target_id"],
    temporal_data["qq_source_ids"],
    temporal_data["site_column"],
    temporal_data["response"],
    labels=temporal_data["qq_labels"],
    target_label="Target",
    title="Normal QQ Plots: NHANES Cholesterol",
    path=FIGURE_DIR / "nhanes_temporal_qq.png",
)

In [ ]:
# Clinicalshift: 2015/2017 no arthritis target and six source cuts
arth_data = prepare_nhanes(
    DATA_DIR,
    source_names=NHANES_ARTHRITIS_SOURCES,
    target_name=NHANES_ARTHRITIS_TARGET,
)
arth_data.update({
    "name": "nhanes_arth",
    "trials": 1000,
    "seed_start": 19000,
    "n_test": 1000,
    "n_t": 30,
    "n_k": 1000,
})
arth_path = RESULT_DIR / "nhanes_arth_results.pkl"

if arth_path.exists():
    with open(arth_path, "rb") as file:
        arth_payload = pickle.load(file)
    assert arth_payload["source_names"] == NHANES_ARTHRITIS_SOURCES
    assert arth_payload["target_name"] == NHANES_ARTHRITIS_TARGET
    assert arth_payload["seeds"] == list(range(19000, 20000))
    assert arth_payload["data_sha256"] == arth_data["data_sha256"]
    validate_payload(arth_payload, arth_data)
    arth_results = arth_payload["results"]
else:
    arth_results = run_trials(
        arth_data,
        trials=1000,
        seed_start=19000,
        jobs=1,
        n_test=1000,
        n_t=30,
        n_k=1000,
        checkpoint_path=RESULT_DIR / "nhanes_arth_checkpoint.pkl",
        checkpoint_every=4,
    )
    save_results(arth_data, arth_results, RESULT_DIR)

arth_summary = summarize_results(arth_results, arth_data["labels"])
arth_plot_summary = pd.read_csv(RESULT_DIR / "nhanes_arth_summary.csv")
pd.testing.assert_frame_equal(
    arth_summary,
    arth_plot_summary,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)
arth_figure = plot_comparison(
    arth_plot_summary,
    "Comparison of methods for data selection: NHANES Cholesterol (clinical shift)",
    y_column="avg_rank",
    y_label="Average Rank of Test MSE (1 = Best)",
    label_offsets={
        "duc": {"D1": (-15, 0), "D3": (-12, -8), "D4": (-12, 8)},
        "neg_kl": {"D3": (8, -8), "D4": (8, 8)},
        "neg_score_x": {"D3": (8, -8), "D4": (-15, 8)},
    },
    path=FIGURE_DIR / "nhanes_arth.png",
)

In [ ]:
# QQ plots for the clinical shift
arth_qq_figure = plot_target_qq(
    arth_data["qq_data"],
    arth_data["target_id"],
    arth_data["qq_source_ids"],
    arth_data["site_column"],
    arth_data["response"],
    labels=arth_data["qq_labels"],
    target_label="Target",
    title="Normal QQ Plots: NHANES Cholesterol",
    path=FIGURE_DIR / "nhanes_arth_qq.png",
)